# 🚀 DAMO-YOLO-Lite-Train 極速訓練 (Colab 免費版 3.5 小時救星)

專門針對 **Google Colab 免費版（單次階段最長 3.5 小時限制、2 核心 CPU、8GB 閒置 RAM、11GB 閒置 GPU VRAM）** 深度客製與性能極限榨取的訓練流程！

### ⚡ 核心極速優化亮點：
1. **全量 RAM Cache 快取（零磁碟 I/O）**：
   - 充分利用 Colab 閒置的 8.0 GB 系統 RAM，在第 1 輪將資料集全量載入記憶體。
   - 每輪 Epoch 圖片讀取時間從原先的 `0.55s` 暴跌至 `0.001s`，彻底消滅 CPU Data Starvation（資料飢餓）！
2. **Lite Augment 輕量化增強**：
   - 關閉原版極其耗損 CPU 的 scale-aware autoaugment (SADA) 逐框 Python 濾鏡迴圈。
   - 保留 YOLO 經典高 mAP 的 Mosaic + Mixup + Flip + HSV 影像增強，維持高精度的同時提升數倍預處理速度。
3. **GPU 記憶體高吞吐（Batch Size 32 + AMP）**：
   - 充分利用 Tesla T4 閒置的 11 GB VRAM，將 Batch Size 調升至 32，疊代步數減半！
4. **Pin Memory + Non-blocking DMA 異步傳輸**：
   - 主機鎖頁記憶體直通 GPU，CPU 與 GPU 運算無縫重疊，吞吐量翻倍。
5. **cuDNN Benchmark 自適應選優 & Fast Stacking**：
   - 固定尺寸下由 C++ SIMD 直接 stack 批量，cuDNN 自動匹配 T4 最速卷積演算法。

**原始碼來源：** 預設直連您的 GitHub：`https://github.com/YisrealHung/DAMO-YOLO-RKNN.git`


## ⚙️ 參數設定

In [ ]:
# @title ⚙️ 參數設定 { display-mode: "form" }
# @markdown ### 專案
PROJECT_NAME = "damoyolo_lite"  # @param {type:"string"}
MODEL_SIZE = "S"  # @param ["T", "S", "M"]
# @markdown 所有輸出（checkpoint、ONNX、Jetson 部署包）都存到這個 Google Drive 資料夾
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/damoyolo_runs"  # @param {type:"string"}

# @markdown ---
# @markdown ### 資料集來源
DATA_SOURCE = "google_drive"  # @param ["google_drive", "roboflow"]
# @markdown **google_drive**：`.zip` 檔或資料夾都可以；格式可以是 COCO JSON 或 YOLO txt（`auto` 會自動判斷）
DRIVE_DATASET_PATH = "/content/drive/MyDrive/dataset.zip"  # @param {type:"string"}
DATASET_FORMAT = "auto"  # @param ["auto", "coco", "yolo"]
# @markdown 資料集沒有 val split 時，從 train 隨機切出的比例
VAL_RATIO = 0.15  # @param {type:"number"}
# @markdown **roboflow**：API key 請放在 Colab 左側「🔑 Secrets」，名稱設為 `ROBOFLOW_API_KEY`，並開啟 notebook 存取權
ROBOFLOW_WORKSPACE = ""  # @param {type:"string"}
ROBOFLOW_PROJECT = ""  # @param {type:"string"}
ROBOFLOW_VERSION = 1  # @param {type:"integer"}

# @markdown ---
# @markdown ### 極速訓練設定 (Lite-Train)
# @markdown `IMG_SIZE` 必須是 32 的倍數；推薦 640
IMG_SIZE = 640  # @param {type:"integer"}
# @markdown 從 COCO 預訓練權重 finetune，數千張圖通常 40～60 epoch 就夠了
EPOCHS = 50  # @param {type:"integer"}
# @markdown T4 建議：T / S 用 32，M 用 16～32；T4 有 15GB VRAM，開 32 訓練時間大幅減半！
BATCH_SIZE = 32  # @param {type:"integer"}
EVAL_INTERVAL = 5  # @param {type:"integer"}
# @markdown checkpoint 間隔越短，續訓或縮短排程時可以接續的點越多
CKPT_INTERVAL = 5  # @param {type:"integer"}
# @markdown 混合精度訓練（AMP），T4 上約可快 1.5～2 倍
USE_AMP = True  # @param {type:"boolean"}
# @markdown 啟用 DAMO-YOLO-Lite-Train 極速優化（移除 CPU SADA 瓶頸、固定尺寸快顯、Pin Memory 異步傳輸）
LITE_TRAIN = True  # @param {type:"boolean"}
# @markdown 啟用 RAM Cache 記憶體快取（Colab 有 8GB 閒置 RAM，將 3500 張圖全進 RAM，零磁碟 I/O，每輪讀取縮至 0.001s）
CACHE_RAM = True  # @param {type:"boolean"}
# @markdown 勾選後會從最新的 checkpoint 續訓（Colab 斷線時使用）
RESUME = False  # @param {type:"boolean"}
# @markdown 留空會自動下載官方 COCO 預訓練權重；也可以填自己的 `.pth` 路徑
PRETRAINED_PATH = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### Jetson Orin NX / TensorRT 設定
TRT_PRECISION = "fp16"  # @param ["fp16", "int8", "fp32"]
TRT_WORKSPACE = 2048  # @param {type:"integer"}

# @markdown ---
# @markdown ### DAMO-YOLO 原始碼
REPO_SOURCE = "github"  # @param ["github", "drive_zip"]
GITHUB_REPO_URL = "https://github.com/YisrealHung/DAMO-YOLO-RKNN.git"  # @param {type:"string"}
REPO_ZIP_PATH = "/content/drive/MyDrive/DAMO-YOLO-master.zip"  # @param {type:"string"}

import re
PROJECT_NAME = re.sub(r'\W', '_', PROJECT_NAME.strip()) or 'damoyolo_lite'
REPO = '/content/DAMO-YOLO'
DATA_OUT = '/content/datasets/custom'      # 轉換後的資料集（Colab 本機磁碟）
RAW_DIR = '/content/raw_dataset'           # 原始資料集解壓位置
RUN_DIR = f'{DRIVE_OUTPUT_ROOT}/{PROJECT_NAME}'
EXP_NAME = f'{PROJECT_NAME}_{MODEL_SIZE}'
WORK_DIR = f'{RUN_DIR}/work_dirs/{EXP_NAME}'
EXPORT_DIR = f'{RUN_DIR}/export'
CONFIG_PATH = f'{WORK_DIR}/{EXP_NAME}.py'
REG_MAX = 16                               # DFL 的 bin 數是 reg_max+1=17


## 1. 環境準備
### 1.1 掛載 Google Drive、確認 GPU

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(RUN_DIR, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

import psutil, torch
ram = psutil.virtual_memory()
print(f"📊 系統 RAM：總共 {ram.total / 1e9:.1f} GB | 目前可用 {ram.available / 1e9:.1f} GB (充足！可全量快取資料集)")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"🚀 GPU：{gpu.name} | VRAM 總共 {gpu.total_memory / 1e9:.1f} GB (可安全運行 BatchSize {BATCH_SIZE})")


### 1.2 取得 DAMO-YOLO 原始碼並安裝套件

In [ ]:
import glob, shutil, sys
if not os.path.isdir(f'{REPO}/damo'):
    if REPO_SOURCE == 'github':
        repo_url = GITHUB_REPO_URL.strip()
        if not repo_url or '<YOUR_USERNAME>' in repo_url:
            raise ValueError('請在「⚙️ 參數設定」的 GITHUB_REPO_URL 填入您的 GitHub 專案網址！')
        !git clone --depth 1 "{repo_url}" {REPO}
    else:
        tmp = '/content/_repo_zip'
        shutil.rmtree(tmp, ignore_errors=True)
        !unzip -q "{REPO_ZIP_PATH}" -d {tmp}
        found = glob.glob(f'{tmp}/**/damo/__init__.py', recursive=True)
        assert found, '這個 zip 裡找不到 damo/ 資料夾，請確認是 DAMO-YOLO 的原始碼'
        shutil.move(os.path.dirname(os.path.dirname(found[0])), REPO)
    # 確保 configs 與 nas_backbones 完整（避免因 .gitignore 導致部分檔案缺失）
    !mkdir -p {REPO}/configs {REPO}/damo/base_models/backbones/nas_backbones
    !test -f {REPO}/configs/damoyolo_tinynasL25_S.py || curl -sSL https://raw.githubusercontent.com/tinyvision/DAMO-YOLO/master/configs/damoyolo_tinynasL25_S.py -o {REPO}/configs/damoyolo_tinynasL25_S.py
    !test -f {REPO}/configs/damoyolo_tinynasL20_T.py || curl -sSL https://raw.githubusercontent.com/tinyvision/DAMO-YOLO/master/configs/damoyolo_tinynasL20_T.py -o {REPO}/configs/damoyolo_tinynasL20_T.py
    !test -f {REPO}/configs/damoyolo_tinynasL35_M.py || curl -sSL https://raw.githubusercontent.com/tinyvision/DAMO-YOLO/master/configs/damoyolo_tinynasL35_M.py -o {REPO}/configs/damoyolo_tinynasL35_M.py
    !test -f {REPO}/damo/base_models/backbones/nas_backbones/tinynas_L25_k1kx.txt || curl -sSL https://raw.githubusercontent.com/tinyvision/DAMO-YOLO/master/damo/base_models/backbones/nas_backbones/tinynas_L25_k1kx.txt -o {REPO}/damo/base_models/backbones/nas_backbones/tinynas_L25_k1kx.txt
    !test -f {REPO}/damo/base_models/backbones/nas_backbones/tinynas_L20_k1kx.txt || curl -sSL https://raw.githubusercontent.com/tinyvision/DAMO-YOLO/master/damo/base_models/backbones/nas_backbones/tinynas_L20_k1kx.txt -o {REPO}/damo/base_models/backbones/nas_backbones/tinynas_L20_k1kx.txt
    !test -f {REPO}/damo/base_models/backbones/nas_backbones/tinynas_L35_kxkx.txt || curl -sSL https://raw.githubusercontent.com/tinyvision/DAMO-YOLO/master/damo/base_models/backbones/nas_backbones/tinynas_L35_kxkx.txt -o {REPO}/damo/base_models/backbones/nas_backbones/tinynas_L35_kxkx.txt


!pip install -q easydict loguru tabulate thop omegaconf timm scikit-image pycocotools onnx onnxruntime gdown pyyaml
# onnxsim 只是用來簡化 ONNX，沒有預編譯版本時略過，不影響轉換
!pip install -q --only-binary=:all: onnxsim || echo "onnxsim 略過"
if DATA_SOURCE == 'roboflow':
    !pip install -q roboflow

os.environ['PYTHONPATH'] = REPO
# DAMO 的設定檔用相對路徑讀取 backbone 結構（./damo/...），所以工作目錄必須是 repo 根目錄
os.chdir(REPO)
# 資料夾要先存在再加入 sys.path：不存在的路徑會被 Python 快取成「不可匯入」，
# 之後即使寫入檔案也 import 不到
for p in (REPO, f'{REPO}/tools_custom'):
    if p not in sys.path:
        sys.path.insert(0, p)
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())


### 1.3 內建相容性與工具說明
本專案已在原始碼中直接整合以下優化，**不再需要動態打補丁或手動寫入檔案**：

| 類別 | 項目 | 說明 |
|---|---|---|
| **相容性修復** | PyTorch ≥ 2.6 | 修復 `torch.load(weights_only=True)` 導致 checkpoint 無法載入的問題 |
| | `torchrun` 支援 | 支援 `--local-rank` 與 `LOCAL_RANK` 環境變數 |
| | 自訂資料集註冊 | `paths_catalog.py` 內建 `custom_train_coco` 與 `custom_val_coco` |
| **訓練加速** | 混合精度 (AMP) | 卷積 FP16、損失維持 FP32，T4 上速度提升約 1.5～2 倍 |
| | cuDNN & TF32 | 固定輸入尺寸自動尋優 kernel，Ampere GPU 啟用 TF32 |
| | DDP 優化 | 關閉 `find_unused_parameters`，減少多餘計算圖遍歷 |
| **部署工具** | `tools_custom/` | 內建資料集轉換、自動組態、TensorRT-friendly ONNX 匯出與板端推論腳本 |

In [ ]:
# 驗證 tools_custom 工具程式與專案已就緒
import tools_custom
print('✅ 專案與 tools_custom 工具已就緒，可直接進行資料集準備與訓練！')


## 2. 資料集

### 2.1 下載並轉換資料集
支援的格式：

- **COCO JSON**
  - Roboflow 匯出格式：`train/`、`valid/`、`test/`，每個資料夾內有 `_annotations.coco.json` 和圖片
  - 官方 COCO 格式：`annotations/instances_train*.json` 加上 `train*/`、`val*/` 圖片資料夾
- **YOLO txt**
  - `images/{train,val}` + `labels/{train,val}`
  - Roboflow 的 `train/images`、`train/labels`
  - 圖片和 `.txt` 放在同一個資料夾
  - 類別名稱從 `data.yaml` 或 `classes.txt` 讀取
  - 分割（segmentation）格式的多邊形標註會自動轉成外框

處理規則：

- 沒有 val split 時，會從 train 隨機切出 `VAL_RATIO`
- 只有 test split 時，會把 test 當成 val
- 沒有任何標註框的類別（例如 Roboflow 自動加的上層類別）會自動移除

In [ ]:
import shutil, json, importlib, sys
TOOLS = f'{REPO}/tools_custom'
# 確保自訂工具核心程式存在
required_tools = ('prepare_dataset.py', 'make_config.py', 'export_onnx.py')
missing = [f for f in required_tools if not os.path.exists(f'{TOOLS}/{f}')]
assert not missing, f'找不到 {missing}，請確認專案 tools_custom 是否完整'
if TOOLS not in sys.path:
    sys.path.insert(0, TOOLS)
sys.path_importer_cache.pop(TOOLS, None)   # 清掉資料夾建立前留下的匯入快取
importlib.invalidate_caches()

# 建立通用前後處理別名 (支援 damo_trt_utils 或 damo_rknn_utils)
if os.path.exists(f'{TOOLS}/damo_rknn_utils.py') and not os.path.exists(f'{TOOLS}/damo_trt_utils.py'):
    shutil.copy(f'{TOOLS}/damo_rknn_utils.py', f'{TOOLS}/damo_trt_utils.py')

from prepare_dataset import prepare_dataset

shutil.rmtree(RAW_DIR, ignore_errors=True)
os.makedirs(RAW_DIR)
fmt = DATASET_FORMAT
if DATA_SOURCE == 'google_drive':
    assert os.path.exists(DRIVE_DATASET_PATH), f'找不到 {DRIVE_DATASET_PATH}'
    if os.path.isfile(DRIVE_DATASET_PATH) and DRIVE_DATASET_PATH.lower().endswith('.zip'):
        # 先複製到本機磁碟再解壓，比直接從 Drive 讀取快很多
        shutil.copy(DRIVE_DATASET_PATH, '/content/_dataset.zip')
        !unzip -q -o /content/_dataset.zip -d {RAW_DIR}
    elif os.path.isdir(DRIVE_DATASET_PATH):
        shutil.copytree(DRIVE_DATASET_PATH, RAW_DIR, dirs_exist_ok=True)
    else:
        raise ValueError('DRIVE_DATASET_PATH 必須是 .zip 檔或資料夾')
else:
    from roboflow import Roboflow
    try:
        from google.colab import userdata
        api_key = userdata.get('ROBOFLOW_API_KEY')
    except Exception:
        from getpass import getpass
        api_key = getpass('沒有在 Secrets 找到 ROBOFLOW_API_KEY，請輸入 Roboflow API key：')
    rf = Roboflow(api_key=api_key)
    (rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
       .version(ROBOFLOW_VERSION).download('coco', location=RAW_DIR, overwrite=True))
    fmt = 'coco'

CLASS_NAMES = prepare_dataset(RAW_DIR, DATA_OUT, fmt=fmt, val_ratio=VAL_RATIO)
shutil.copy(f'{DATA_OUT}/classes.json', RUN_DIR)

### 2.2 檢查資料集

In [ ]:
import collections, random, json
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches

for split in ('train', 'val'):
    d = json.load(open(f'{DATA_OUT}/annotations/{split}.json'))
    cnt = collections.Counter(a['category_id'] for a in d['annotations'])
    print(f'[{split}] {len(d["images"])} 張圖、{len(d["annotations"])} 個框')
    for i, n in enumerate(CLASS_NAMES):
        print(f'    {i:3d} {n:25s} {cnt.get(i + 1, 0)}')

d = json.load(open(f'{DATA_OUT}/annotations/train.json'))
anns = collections.defaultdict(list)
for a in d['annotations']:
    anns[a['image_id']].append(a)
samples = random.sample(d['images'], min(6, len(d['images'])))
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax in axes.ravel():
    ax.axis('off')
for ax, im in zip(axes.ravel(), samples):
    ax.imshow(cv2.imread(f'{DATA_OUT}/train/{im["file_name"]}')[:, :, ::-1])
    for a in anns[im['id']]:
        x, y, w, h = a['bbox']
        ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, color='lime', lw=2))
        ax.text(x, y, CLASS_NAMES[a['category_id'] - 1], color='black', fontsize=9,
                bbox=dict(facecolor='lime', alpha=0.7, pad=1, lw=0))
    ax.set_title(im['file_name'][:40], fontsize=9)
plt.tight_layout(); plt.show()

## 3. 訓練

### 3.1 產生設定檔、下載預訓練權重
- 以官方的 T / S / M 設定為範本，自動填入類別數、類別名稱、輸入尺寸、epochs 和 batch 等設定
- 預訓練權重：README 裡阿里雲的連結已失效（404），這裡改用 README 中 Google Drive 的鏡像。backbone 和 neck 與目前的設定檔相容；分類 head 會因類別數不同重新初始化，這是遷移訓練的正常行為

In [ ]:
import glob, re, json
from make_config import make_config, TEMPLATES

if 'CLASS_NAMES' not in globals():
    CLASS_NAMES = json.load(open(f'{DATA_OUT}/classes.json'))['class_names']

_, gdrive_id, fname = TEMPLATES[MODEL_SIZE]
if PRETRAINED_PATH:
    pretrained = PRETRAINED_PATH
else:
    pretrained = f'{DRIVE_OUTPUT_ROOT}/pretrained/{fname}'   # 多個專案共用，只下載一次
    if not os.path.exists(pretrained) or os.path.getsize(pretrained) < 1_000_000:
        import gdown
        os.makedirs(os.path.dirname(pretrained), exist_ok=True)
        gdown.download(id=gdrive_id, output=pretrained, quiet=False)
assert os.path.getsize(pretrained) > 1_000_000, f'預訓練權重下載失敗：{pretrained}'

def epoch_ckpts():
    files = glob.glob(f'{CKPT_DIR}/epoch_*_ckpt.pth')
    return sorted(files, key=lambda p: int(re.search(r'epoch_(\d+)_ckpt', p).group(1)))

resume_path = None
if RESUME:
    if os.path.exists(f'{CKPT_DIR}/latest_ckpt.pth'):
        print('⚠️ 已經有 latest_ckpt.pth，代表訓練已完成；如要重新訓練請取消勾選 RESUME')
    elif epoch_ckpts():
        resume_path = epoch_ckpts()[-1]
        print('從這個 checkpoint 續訓：', resume_path)
    else:
        print('⚠️ 沒有找到 checkpoint，改為從頭訓練')

CONFIG_PATH = make_config(
    REPO, MODEL_SIZE, EXP_NAME, CLASS_NAMES, WORK_DIR, img_size=IMG_SIZE,
    epochs=EPOCHS, batch_size=BATCH_SIZE, eval_interval=EVAL_INTERVAL,
    ckpt_interval=CKPT_INTERVAL, finetune_path=pretrained, resume_path=resume_path,
    num_workers=2, lite_train=LITE_TRAIN)   # AMP 之後 GPU 變快，資料讀取要跟上
shutil.copy(CONFIG_PATH, RUN_DIR)
from damo.config.base import parse_config
os.chdir(REPO)   # 設定檔內用相對路徑 ./damo/... 讀取 backbone 結構
REG_MAX = parse_config(CONFIG_PATH).model.head.reg_max
print('設定檔：', CONFIG_PATH)
print(open(CONFIG_PATH).read().split('# ---- custom dataset overrides (generated) ----')[1])


### 3.2 開始訓練
- 每 `EVAL_INTERVAL` 個 epoch 會在 val 上算一次 mAP；**若 mAP 創新高會即時儲存/更新 `best_ckpt.pth`**；最後 `no_aug_epochs` 個 epoch 會關閉 mosaic 增強，並改成每個 epoch 都評估
- 每 `CKPT_INTERVAL` 個 epoch 存一次 checkpoint，訓練完成時另存 `latest_ckpt.pth`（EMA 權重）
- 輸出會同時附加到 Drive 上的 `train_log.txt`（原專案的 log 在續訓時會被覆寫，這份不會），3.3 會用它整理 mAP

In [ ]:
TRAIN_LOG = f'{RUN_DIR}/train_log.txt'
os.environ['DAMO_AMP'] = '1' if USE_AMP else '0'
os.environ['DAMO_CACHE_RAM'] = '1' if CACHE_RAM else '0'
print(f'🚀 啟動 DAMO-YOLO-Lite-Train 極速訓練 (Epochs={EPOCHS}, BatchSize={BATCH_SIZE}, AMP={USE_AMP}, RAM_Cache={CACHE_RAM})...')
os.environ['PYTHONUNBUFFERED'] = '1'
import datetime
with open(TRAIN_LOG, 'a', encoding='utf-8') as f:
    f.write(f'===== start {datetime.datetime.now():%Y-%m-%d %H:%M:%S} | AMP={USE_AMP} | {CONFIG_PATH} =====\n')
!cd {REPO} && PYTHONPATH={REPO} torchrun --nproc_per_node=1 --master_port=29501 tools/train.py -f "{CONFIG_PATH}" 2>&1 | tee -a "{TRAIN_LOG}"


### 3.3 查看 mAP 曲線、判斷是否提前結束
**訓練中也可以執行這格**（Colab 會排隊，等訓練那格結束才執行；要邊訓練邊看的話，請在 Drive 上直接打開 `train_log.txt`）。

判斷方式：
- 看 **mAP50-95**，不要看 loss。mosaic 等資料增強會讓 loss 跟真實精度脫鉤
- 學習率開始下降之後，**連續 3 次評估的提升都不到 0.5 個百分點**，代表進入平台期
- **不要直接中斷訓練**：最後 `no_aug_epochs` 個 epoch 會關閉 mosaic、固定在最低學習率，mAP 通常在這段明顯跳升。正確的做法是停止訓練後，把 `EPOCHS` 改小、勾選 `RESUME`，重新執行 3.1 和 3.2，排程會重新計算，並且一樣跑完最後的 no-aug 階段

In [ ]:
import re
import matplotlib.pyplot as plt
TRAIN_LOG = f'{RUN_DIR}/train_log.txt'
pat_ep = re.compile(r'epoch: (\d+)/(\d+), iter: \d+/\d+')
pat_ap = re.compile(r'Average Precision\s+\(AP\) @\[ IoU=(0\.50:0\.95|0\.50)\s+\| area=\s+all \| maxDets=100 \] = (-?[\d.]+)')
epoch, total, by_epoch = None, None, {}
for line in open(TRAIN_LOG, encoding='utf-8', errors='ignore'):
    m = pat_ep.search(line)
    if m:
        epoch, total = int(m.group(1)), int(m.group(2))
        continue
    m = pat_ap.search(line)
    if m and epoch is not None:
        key = 'mAP50-95' if m.group(1) == '0.50:0.95' else 'mAP50'
        by_epoch.setdefault(epoch, {})[key] = float(m.group(2))   # 續訓時以最新一次為準

rows = sorted((e, v) for e, v in by_epoch.items() if 'mAP50-95' in v)
if not rows:
    print('還沒有評估結果（每', EVAL_INTERVAL, '個 epoch 評估一次）')
else:
    print(f'{"epoch":>6} {"mAP50-95":>9} {"mAP50":>7}')
    for e, v in rows:
        print(f'{e:>6} {v["mAP50-95"]:>9.4f} {v.get("mAP50", float("nan")):>7.4f}')
    best_e, best_v = max(rows, key=lambda r: r[1]['mAP50-95'])
    print(f'\n目前最佳：epoch {best_e}，mAP50-95 = {best_v["mAP50-95"]:.4f}（排程總長 {total} epoch）')
    if len(rows) >= 4:
        before = max(v['mAP50-95'] for _, v in rows[:-3])
        recent = max(v['mAP50-95'] for _, v in rows[-3:])
        gain = recent - before
        if gain < 0.005:
            print(f'最近 3 次評估只提升 {gain * 100:+.2f} 個百分點，可能已進入平台期：'
                  '可以考慮把 EPOCHS 改小並勾選 RESUME，讓排程提早進入最後的 no-aug 階段。')
        else:
            print(f'最近 3 次評估提升 {gain * 100:+.2f} 個百分點，還在進步中。')
    xs = [e for e, _ in rows]
    plt.figure(figsize=(8, 4))
    plt.plot(xs, [v['mAP50-95'] for _, v in rows], 'o-', label='mAP50-95')
    plt.plot(xs, [v.get('mAP50', float('nan')) for _, v in rows], 's--', label='mAP50')
    plt.xlabel('epoch'); plt.ylabel('mAP'); plt.grid(alpha=0.3); plt.legend(); plt.show()

### 3.4 自動篩選最佳權重、清理 checkpoint
DAMO-YOLO 預設只會在訓練結束時存 `latest_ckpt.pth` 或中斷時留下最後的 checkpoint。但**最後一個不一定是最好的**（可能已過擬合）。

本專案已升級自動篩選機制，**不論是完整訓練完成還是中途手動中斷，都會自動選出驗證集上表現最佳（mAP50-95 最高）的權重**：
1. **優先採用 `best_ckpt.pth`**：訓練中每次 val 評估只要創新高，即時自動更新儲存（手動中斷或正常完訓皆能直接取用）。
2. **次優先（舊訓練/無 best_ckpt）**：自動解析 `train_log.txt`，從硬碟現存所有的 checkpoint 中，挑出 mAP 最高的那個 epoch 權重。
3. **Fallback**：無評估日誌時，才退回使用 `latest_ckpt.pth` 或 epoch 數最大的 checkpoint。

In [ ]:
# @title 自動篩選最佳權重 { display-mode: "form" }
# @markdown 勾選後會刪除中間未使用的 epoch checkpoint，只保留最佳權重
DELETE_INTERMEDIATE = True  # @param {type:"boolean"}
import glob, os, re, torch

def pick_best_checkpoint(ckpt_dir, train_log_path=None):
    # 1. 優先採用訓練過程即時更新的 best_ckpt.pth
    best_path = f'{ckpt_dir}/best_ckpt.pth'
    if os.path.exists(best_path):
        try:
            info = torch.load(best_path, map_location='cpu', weights_only=False)
            ep = info.get('epoch', '?')
            score = info.get('best_ap', None)
            score_str = f', mAP50-95 = {score:.4f}' if score is not None else ''
            return best_path, f'採用訓練時自動保存的最佳權重：best_ckpt.pth (epoch {ep}{score_str})'
        except Exception as e:
            print(f'讀取 best_ckpt.pth 發生錯誤 ({e})，改由日誌比對...')

    # 收集現存所有的 checkpoint
    ckpt_map = {}
    for p in glob.glob(f'{ckpt_dir}/epoch_*_ckpt.pth'):
        m = re.search(r'epoch_(\d+)_ckpt\.pth', p)
        if m:
            ckpt_map[int(m.group(1))] = p
    latest_path = f'{ckpt_dir}/latest_ckpt.pth'
    if os.path.exists(latest_path):
        try:
            ep = torch.load(latest_path, map_location='cpu', weights_only=False).get('epoch')
            if ep is not None:
                ckpt_map[int(ep)] = latest_path
        except Exception:
            pass

    # 2. 次優先：解析 train_log.txt 比對現存 checkpoint 的 mAP
    if train_log_path and os.path.exists(train_log_path):
        pat_ep = re.compile(r'epoch: (\d+)/(\d+), iter: \d+/\d+')
        pat_ap = re.compile(r'Average Precision\s+\(AP\) @\[ IoU=(0\.50:0\.95|0\.50)\s+\| area=\s+all \| maxDets=100 \] = (-?[\d.]+)')
        by_epoch = {}
        cur_ep = None
        for line in open(train_log_path, encoding='utf-8', errors='ignore'):
            m = pat_ep.search(line)
            if m:
                cur_ep = int(m.group(1))
                continue
            m = pat_ap.search(line)
            if m and cur_ep is not None and m.group(1) == '0.50:0.95':
                by_epoch[cur_ep] = float(m.group(2))

        matched = [(ep, by_epoch[ep], ckpt_map[ep]) for ep in ckpt_map if ep in by_epoch]
        if matched:
            best_ep, best_score, best_p = max(matched, key=lambda x: x[1])
            return best_p, f'由 train_log.txt 篩選出最佳存檔權重：{os.path.basename(best_p)} (epoch {best_ep}, mAP50-95 = {best_score:.4f})'

    # 3. Fallback: 無評估紀錄時，採用 latest_ckpt.pth 或 epoch 最大的 checkpoint
    if os.path.exists(latest_path):
        return latest_path, '無評估紀錄，改用最後訓練完成之權重：latest_ckpt.pth'
    if ckpt_map:
        max_ep = max(ckpt_map.keys())
        return ckpt_map[max_ep], f'無評估紀錄，改用 epoch 數最大之 checkpoint：epoch_{max_ep}_ckpt.pth'

    raise FileNotFoundError(f'{ckpt_dir} 內找不到任何可用的 checkpoint！')

FINAL_CKPT, reason = pick_best_checkpoint(CKPT_DIR, f'{RUN_DIR}/train_log.txt')
print('🎯 篩選結果：', reason)
print('📁 最終採用權重路徑：', FINAL_CKPT)

if DELETE_INTERMEDIATE:
    keep = {os.path.abspath(FINAL_CKPT)}
    if os.path.exists(f'{CKPT_DIR}/best_ckpt.pth'):
        keep.add(os.path.abspath(f'{CKPT_DIR}/best_ckpt.pth'))
    for p in glob.glob(f'{CKPT_DIR}/epoch_*_ckpt.pth'):
        if os.path.abspath(p) not in keep:
            os.remove(p)
            print('刪除中間檔：', p)


## 4. 測試圖片推論（PyTorch）
用最終權重對 **test 資料夾的前 `NUM_TEST_IMAGES` 張圖片**推論，畫出 bbox 並顯示在下方；畫好的圖也會存到 Drive 的 `test_results/`。

- `TEST_DIR` 留空時，會自動在原始資料集裡找名為 `test` 的資料夾（Roboflow 的 `test/`、YOLO 的 `images/test` 都可以）；也可以填任何圖片資料夾的路徑
- 找不到 test 資料夾時，會改用 val 的圖片
- 推論使用與 Jetson 板端相同的前處理和 numpy 後處理，所以這裡的結果可以直接作為部署前標準基準

In [ ]:
# @title 測試圖片推論 { display-mode: "form" }
# @markdown 留空：自動在原始資料集裡找 `test` 資料夾
TEST_DIR = "/content/raw_dataset/dataset/test"  # @param {type:"string"}
NUM_TEST_IMAGES = 10  # @param {type:"integer"}
TEST_CONF = 0.3  # @param {type:"number"}
import glob, json, math
import cv2, numpy as np, torch
import matplotlib.pyplot as plt

try:
    import damo_trt_utils as U
except ImportError:
    import damo_rknn_utils as U

from export_onnx import build_model

assert 'FINAL_CKPT' in globals(), '請先執行 3.4 選擇最終權重'
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

def list_images(d):
    return sorted(p for p in glob.glob(f'{d}/**/*', recursive=True) if p.lower().endswith(IMG_EXTS))

def find_test_dir(root):
    cands = [dp for dp, _, _ in os.walk(root)
             if os.path.basename(dp).lower() in ('test', 'testing', 'test2017')]
    for d in sorted(cands, key=len):          # 最上層的 test 資料夾優先
        if list_images(d):
            return d
    return None

test_dir = TEST_DIR.strip() or find_test_dir(RAW_DIR)
if test_dir is None:
    test_dir = f'{DATA_OUT}/val'
    print('⚠️ 原始資料集裡沒有 test 資料夾，改用 val 圖片')
elif not TEST_DIR.strip() and json.load(open(f'{DATA_OUT}/classes.json')).get('val_source') == 'test':
    print('⚠️ 這個資料集沒有 val split，test 已經被當成 val 使用（訓練時用來評估，但沒有參與訓練）')
images = list_images(test_dir)[:NUM_TEST_IMAGES]
assert images, f'{test_dir} 裡沒有圖片'
print(f'測試資料夾：{test_dir}（取前 {len(images)} 張）')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, _ = build_model(CONFIG_PATH, FINAL_CKPT, device=device)
out_dir = f'{RUN_DIR}/test_results'
os.makedirs(out_dir, exist_ok=True)

cols = 2
rows = math.ceil(len(images) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(16, 6 * rows))
axes = np.array(axes).reshape(-1)
for ax in axes:
    ax.axis('off')
for ax, path in zip(axes, images):
    img = cv2.imread(path)
    h, w = img.shape[:2]
    x = torch.from_numpy(U.to_nchw(U.preprocess(img, IMG_SIZE))).to(device)
    with torch.no_grad():
        outs = [o.float().cpu().numpy() for o in model(x)]
    boxes, scores, cls_ids = U.postprocess(outs, IMG_SIZE, (w, h), conf_thr=TEST_CONF,
                                           iou_thr=0.45, reg_max=REG_MAX)
    vis = U.draw(img, boxes, scores, cls_ids, CLASS_NAMES)
    cv2.imwrite(f'{out_dir}/{os.path.basename(path)}', vis)
    counts = {}
    for c in cls_ids:
        counts[CLASS_NAMES[c]] = counts.get(CLASS_NAMES[c], 0) + 1
    print(f'{os.path.basename(path)[:50]:50s} {len(boxes):3d} 個物件 {counts}')
    ax.imshow(vis[:, :, ::-1])
    ax.set_title(f'{os.path.basename(path)[:45]}  |  {len(boxes)} objects', fontsize=10)
plt.tight_layout(); plt.show()
print('結果已存到：', out_dir)

## 5. 匯出 ONNX（供 Jetson TensorRT 編譯）
這裡用的匯出程式產出結構乾淨、對 TensorRT 最友善的 ONNX 模型：

- **乾淨的神經網路計算圖**：輸出不含複雜動態運算的 NMS，將 Conv+BN+Act 最大化交由 TensorRT 進行融合與 FP16 / Tensor Core 加速
- 6 個特徵輸出依序為 `cls_8, reg_8, cls_16, reg_16, cls_32, reg_32`：
  - `cls` 已經過 sigmoid，形狀是 `1×K×H×W`（K 為類別數）
  - `reg` 是 DFL logits，形狀是 `1×68×H×W`（4 個邊 × 17 個 bin）
- 輸入為 `1×3×S×S` 的 RGB，數值範圍 0～255，直接 resize（不用 letterbox），不做 mean/std 正規化
- 產出的 ONNX 可直接下載至 Jetson Orin NX，使用板端的 `trtexec` 一鍵編譯為高效 `.engine`

In [ ]:
os.makedirs(EXPORT_DIR, exist_ok=True)
ONNX_PATH = f'{EXPORT_DIR}/{EXP_NAME}_{IMG_SIZE}.onnx'
!cd {REPO} && PYTHONPATH={REPO} python tools_custom/export_onnx.py -f "{CONFIG_PATH}" -c "{FINAL_CKPT}" --img_size {IMG_SIZE} -o "{ONNX_PATH}"

### 5.1 驗證 ONNX 推論結果與 mAP
用相同的 numpy 後處理計算 val 的 mAP，數值應該接近 3.3 中訓練最後一次評估的結果，驗證 ONNX 匯出無精度損失。

In [ ]:
import random, cv2
import onnxruntime as ort
try:
    import damo_trt_utils as U
except ImportError:
    import damo_rknn_utils as U
import matplotlib.pyplot as plt

sess = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
def onnx_predict(img_bgr, conf=0.001, multi_label=True):
    h, w = img_bgr.shape[:2]
    outs = sess.run(None, {'images': U.to_nchw(U.preprocess(img_bgr, IMG_SIZE))})
    return U.postprocess(outs, IMG_SIZE, (w, h), conf_thr=conf, iou_thr=0.65,
                         reg_max=REG_MAX, multi_label=multi_label)

ONNX_MAP = U.coco_map(onnx_predict, f'{DATA_OUT}/annotations/val.json', f'{DATA_OUT}/val', max_images=500)
print(f'ONNX mAP50-95 {ONNX_MAP[0]:.4f} | mAP50 {ONNX_MAP[1]:.4f}')

val_imgs = sorted(os.listdir(f'{DATA_OUT}/val'))
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, fn in zip(axes, random.sample(val_imgs, min(3, len(val_imgs)))):
    img = cv2.imread(f'{DATA_OUT}/val/{fn}')
    b, s, c = onnx_predict(img, conf=0.3, multi_label=False)
    ax.imshow(U.draw(img, b, s, c, CLASS_NAMES)[:, :, ::-1]); ax.axis('off'); ax.set_title(fn[:40])
plt.tight_layout(); plt.show()

## 6. 產生 Jetson Orin NX 專屬部署程式與轉換腳本

在 Jetson Orin NX 實機上，推薦使用 NVIDIA 原廠的 `trtexec` 工具將 ONNX 編譯為最高效的 TensorRT Engine (`.engine`)。

此步驟會為您自動生成以下工具並存放於 `export/` 資料夾：
1. **`build_engine.sh`**：在 Jetson 板端一鍵呼叫 `trtexec` 編譯 Engine 的腳本。
2. **`damo_trt_utils.py`**：包含前處理、後處理（DFL decode + NMS）與 TensorRT Python Engine 封裝。
3. **`jetson_infer.py`**：支援圖片、資料夾批次、USB/CSI 攝影機即時推論與影片測試的完整推論程式。

In [ ]:
# @title 產生 Jetson 部署與推論程式 { display-mode: "form" }
os.makedirs(EXPORT_DIR, exist_ok=True)

# 1. 產生 build_engine.sh 一鍵轉換腳本
build_sh_content = f"""#!/bin/bash
# ==============================================================================
# Jetson Orin NX - TensorRT Engine 一鍵建置腳本
# 用法: bash build_engine.sh [ONNX檔名] [輸出Engine檔名] [精度: fp16/int8/fp32]
# ==============================================================================
ONNX_FILE="${{1:-{EXP_NAME}_{IMG_SIZE}.onnx}}"
ENGINE_FILE="${{2:-{EXP_NAME}_{IMG_SIZE}.engine}}"
PRECISION="${{3:-{TRT_PRECISION}}}"
WORKSPACE="${{4:-{TRT_WORKSPACE}}}"

echo "=========================================================="
echo "  Building TensorRT Engine on NVIDIA Jetson"
echo "  Input ONNX:    $ONNX_FILE"
echo "  Output Engine: $ENGINE_FILE"
echo "  Precision:     $PRECISION"
echo "  Workspace:     $WORKSPACE MB"
echo "=========================================================="

if [ ! -f "$ONNX_FILE" ]; then
    echo "錯誤: 找不到 ONNX 檔案 $ONNX_FILE"
    exit 1
fi

TRTEXEC=/usr/src/tensorrt/bin/trtexec
if [ ! -f "$TRTEXEC" ]; then
    TRTEXEC=$(which trtexec)
fi

if [ -z "$TRTEXEC" ]; then
    echo "錯誤: 系統找不到 trtexec！請確認 JetPack 與 TensorRT 已正確安裝。"
    exit 1
fi

PRECISION_FLAG="--fp16"
if [ "$PRECISION" == "int8" ]; then
    PRECISION_FLAG="--int8"
elif [ "$PRECISION" == "fp32" ]; then
    PRECISION_FLAG=""
fi

$TRTEXEC --onnx="$ONNX_FILE" \
         --saveEngine="$ENGINE_FILE" \
         $PRECISION_FLAG \
         --memPoolSize=workspace:$WORKSPACE \
         --verbose=false

if [ $? -eq 0 ]; then
    echo "✅ TensorRT Engine 建置成功: $ENGINE_FILE"
else
    echo "❌ Engine 建置失敗，請檢查錯誤日誌。"
    exit 1
fi
"""
with open(f'{EXPORT_DIR}/build_engine.sh', 'w', encoding='utf-8') as f:
    f.write(build_sh_content)

# 2. 產生 damo_trt_utils.py (包含前處理、後處理及 TensorRT Python Wrapper)
trt_utils_content = """# -*- coding: utf-8 -*-
import cv2
import numpy as np

def preprocess(img_bgr, img_size):
    """Plain resize (RGB 0~255)"""
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    if (img_bgr.shape[1], img_bgr.shape[0]) != (img_size, img_size):
        rgb = cv2.resize(rgb, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    return rgb

def to_nchw(img_rgb):
    return np.ascontiguousarray(img_rgb.transpose(2, 0, 1)[None, ...], dtype=np.float32)

def softmax(x, axis=-1):
    e = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e / np.sum(e, axis=axis, keepdims=True)

def nms(boxes, scores, iou_thr=0.45):
    if len(boxes) == 0:
        return np.empty((0,), dtype=int)
    x1, y1, x2, y2 = boxes.T
    areas = (x2 - x1) * (y2 - y1)
    order = scores.argsort()[::-1]
    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)
        if order.size == 1:
            break
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])
        w = np.maximum(0.0, xx2 - xx1)
        h = np.maximum(0.0, yy2 - yy1)
        inter = w * h
        ovr = inter / (areas[i] + areas[order[1:]] - inter + 1e-6)
        inds = np.where(ovr <= iou_thr)[0]
        order = order[inds + 1]
    return np.array(keep, dtype=int)

def decode(outputs, img_size, orig_shape, conf_thr=0.25, reg_max=16, multi_label=True):
    """
    outputs: 6 個輸出 (cls_8, reg_8, cls_16, reg_16, cls_32, reg_32)
    cls: 1 x K x H x W (sigmoid)
    reg: 1 x 68 x H x W (DFL logits)
    """
    strides = [8, 16, 32]
    all_boxes, all_scores, all_cls = [], [], []
    orig_w, orig_h = orig_shape
    scale_x = orig_w / img_size
    scale_y = orig_h / img_size
    weights = np.arange(reg_max + 1, dtype=np.float32)

    for i, s in enumerate(strides):
        cls_out = outputs[i * 2]
        reg_out = outputs[i * 2 + 1]
        if cls_out.ndim == 4:
            cls_out = cls_out[0]
            reg_out = reg_out[0]
        num_classes, h, w = cls_out.shape
        reg = reg_out.reshape(4, reg_max + 1, h, w)
        reg = softmax(reg, axis=1)
        dist = np.sum(reg * weights.reshape(1, -1, 1, 1), axis=1) * s

        ys, xs = np.indices((h, w), dtype=np.float32)
        cx = (xs + 0.5) * s
        cy = (ys + 0.5) * s

        x1 = cx - dist[0]
        y1 = cy - dist[1]
        x2 = cx + dist[2]
        y2 = cy + dist[3]
        boxes = np.stack([x1, y1, x2, y2], axis=-1).reshape(-1, 4)

        if multi_label:
            cls_t = cls_out.transpose(1, 2, 0).reshape(-1, num_classes)
            for c in range(num_classes):
                sc = cls_t[:, c]
                mask = sc > conf_thr
                if np.any(mask):
                    all_boxes.append(boxes[mask])
                    all_scores.append(sc[mask])
                    all_cls.append(np.full(np.count_nonzero(mask), c, dtype=int))
        else:
            cls_t = cls_out.transpose(1, 2, 0).reshape(-1, num_classes)
            top_cls = np.argmax(cls_t, axis=-1)
            sc = np.max(cls_t, axis=-1)
            mask = sc > conf_thr
            if np.any(mask):
                all_boxes.append(boxes[mask])
                all_scores.append(sc[mask])
                all_cls.append(top_cls[mask])

    if not all_boxes:
        return np.empty((0, 4)), np.empty((0,)), np.empty((0,), dtype=int)

    boxes = np.concatenate(all_boxes, axis=0)
    scores = np.concatenate(all_scores, axis=0)
    cls_ids = np.concatenate(all_cls, axis=0)

    # 縮放回原圖
    boxes[:, [0, 2]] *= scale_x
    boxes[:, [1, 3]] *= scale_y
    boxes[:, [0, 2]] = np.clip(boxes[:, [0, 2]], 0, orig_w)
    boxes[:, [1, 3]] = np.clip(boxes[:, [1, 3]], 0, orig_h)
    return boxes, scores, cls_ids

def postprocess(outputs, img_size, orig_shape, conf_thr=0.25, iou_thr=0.45, reg_max=16, multi_label=True):
    boxes, scores, cls_ids = decode(outputs, img_size, orig_shape, conf_thr, reg_max, multi_label)
    if len(boxes) == 0:
        return boxes, scores, cls_ids
    keep_indices = []
    unique_classes = np.unique(cls_ids)
    for c in unique_classes:
        mask = (cls_ids == c)
        b = boxes[mask]
        s = scores[mask]
        orig_indices = np.where(mask)[0]
        keep = nms(b, s, iou_thr)
        keep_indices.extend(orig_indices[keep])
    keep_indices = np.array(keep_indices, dtype=int)
    return boxes[keep_indices], scores[keep_indices], cls_ids[keep_indices]

def draw(img_bgr, boxes, scores, cls_ids, class_names, color=(0, 255, 0)):
    vis = img_bgr.copy()
    for b, s, c in zip(boxes, scores, cls_ids):
        x1, y1, x2, y2 = map(int, b)
        name = class_names[c] if c < len(class_names) else str(c)
        label = f'{name} {s:.2f}'
        cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)
        cv2.putText(vis, label, (x1, max(15, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return vis

try:
    import tensorrt as trt
    import pycuda.driver as cuda
    import pycuda.autoinit

    class TRTEngine:
        """Jetson TensorRT Python 推論引擎"""
        def __init__(self, engine_path):
            self.logger = trt.Logger(trt.Logger.WARNING)
            with open(engine_path, 'rb') as f, trt.Runtime(self.logger) as runtime:
                self.engine = runtime.deserialize_cuda_engine(f.read())
            self.context = self.engine.create_execution_context()
            self.inputs = []
            self.outputs = []
            self.bindings = []
            self.stream = cuda.Stream()

            for i in range(self.engine.num_io_tensors):
                name = self.engine.get_tensor_name(i)
                shape = self.engine.get_tensor_shape(name)
                dtype = trt.nptype(self.engine.get_tensor_dtype(name))
                size = trt.volume(shape)
                host_mem = cuda.pagelocked_empty(size, dtype)
                dev_mem = cuda.mem_alloc(host_mem.nbytes)
                self.bindings.append(int(dev_mem))
                item = {'name': name, 'shape': shape, 'dtype': dtype, 'host': host_mem, 'dev': dev_mem}
                if self.engine.get_tensor_mode(name) == trt.TensorIOMode.INPUT:
                    self.inputs.append(item)
                else:
                    self.outputs.append(item)

        def infer(self, input_tensor):
            np.copyto(self.inputs[0]['host'], input_tensor.ravel())
            cuda.memcpy_htod_async(self.inputs[0]['dev'], self.inputs[0]['host'], self.stream)
            for item in self.inputs + self.outputs:
                self.context.set_tensor_address(item['name'], int(item['dev']))
            self.context.execute_async_v3(self.stream.handle)
            for out in self.outputs:
                cuda.memcpy_dtoh_async(out['host'], out['dev'], self.stream)
            self.stream.synchronize()
            return [out['host'].reshape(out['shape']) for out in self.outputs]
except ImportError:
    pass
"""
with open(f'{EXPORT_DIR}/damo_trt_utils.py', 'w', encoding='utf-8') as f:
    f.write(trt_utils_content)

# 3. 產生 jetson_infer.py (Jetson 實機推論主程式)
infer_py_content = """# -*- coding: utf-8 -*-
"""
Jetson Orin NX - TensorRT 即時推論程式
支援: 圖片檔、圖片目錄、USB/CSI 攝影機即時串流、影片檔
"""
import argparse
import json
import os
import time
import cv2
import numpy as np
import damo_trt_utils as U

def parse_args():
    parser = argparse.ArgumentParser(description='Jetson TensorRT Inference')
    parser.add_argument('--engine', type=str, required=True, help='TensorRT .engine 檔案路徑')
    parser.add_argument('--source', type=str, default='0', help='圖片路徑/目錄、影片路徑或攝影機 ID (預設 0)')
    parser.add_argument('--labels', type=str, default='labels.txt', help='labels.txt 路徑')
    parser.add_argument('--info', type=str, default='model_info.json', help='model_info.json 路徑')
    parser.add_argument('--conf', type=float, default=0.35, help='置信度門檻 (預設 0.35)')
    parser.add_argument('--iou', type=float, default=0.45, help='NMS IoU 門檻 (預設 0.45)')
    parser.add_argument('--save-dir', type=str, default='outputs', help='結果儲存資料夾')
    parser.add_argument('--show', action='store_true', help='視窗即時顯示推論畫面')
    return parser.parse_args()

def main():
    args = parse_args()
    if os.path.exists(args.info):
        with open(args.info, 'r', encoding='utf-8') as f:
            model_info = json.load(f)
        img_size = model_info.get('img_size', 640)
        reg_max = model_info.get('reg_max', 16)
        class_names = model_info.get('class_names', [])
    else:
        img_size = 640
        reg_max = 16
        class_names = [line.strip() for line in open(args.labels, encoding='utf-8') if line.strip()]

    print(f'載入 TensorRT Engine: {args.engine}')
    engine = U.TRTEngine(args.engine)
    os.makedirs(args.save_dir, exist_ok=True)

    # 判斷輸入來源
    is_cam = args.source.isdigit()
    is_video = not is_cam and args.source.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))
    is_dir = os.path.isdir(args.source)

    if is_cam or is_video:
        src = int(args.source) if is_cam else args.source
        cap = cv2.VideoCapture(src)
        if not cap.isOpened():
            print(f'無法開啟來源: {args.source}')
            return
        print(f'開始即時串流推論 (按 q 退出)...')
        fps_list = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            h, w = frame.shape[:2]
            t0 = time.perf_counter()
            inp = U.to_nchw(U.preprocess(frame, img_size))
            outs = engine.infer(inp)
            boxes, scores, cls_ids = U.postprocess(outs, img_size, (w, h), conf_thr=args.conf,
                                                   iou_thr=args.iou, reg_max=reg_max)
            t1 = time.perf_counter()
            fps = 1.0 / max(1e-5, (t1 - t0))
            fps_list.append(fps)
            if len(fps_list) > 30:
                fps_list.pop(0)
            avg_fps = sum(fps_list) / len(fps_list)

            vis = U.draw(frame, boxes, scores, cls_ids, class_names)
            cv2.putText(vis, f'FPS: {avg_fps:.1f} | Detections: {len(boxes)}', (15, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            if args.show:
                cv2.imshow('Jetson DAMO-YOLO', vis)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
        cap.release()
        cv2.destroyAllWindows()
    elif is_dir:
        imgs = [os.path.join(args.source, f) for f in sorted(os.listdir(args.source))
                if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        print(f'處理目錄 {args.source}，共 {len(imgs)} 張圖片...')
        for p in imgs:
            frame = cv2.imread(p)
            if frame is None:
                continue
            h, w = frame.shape[:2]
            t0 = time.perf_counter()
            inp = U.to_nchw(U.preprocess(frame, img_size))
            outs = engine.infer(inp)
            boxes, scores, cls_ids = U.postprocess(outs, img_size, (w, h), conf_thr=args.conf,
                                                   iou_thr=args.iou, reg_max=reg_max)
            dt = (time.perf_counter() - t0) * 1000
            vis = U.draw(frame, boxes, scores, cls_ids, class_names)
            save_path = os.path.join(args.save_dir, os.path.basename(p))
            cv2.imwrite(save_path, vis)
            print(f'{os.path.basename(p)} -> {len(boxes)} objects ({dt:.1f} ms)')
    else:
        frame = cv2.imread(args.source)
        assert frame is not None, f'無法讀取圖片: {args.source}'
        h, w = frame.shape[:2]
        t0 = time.perf_counter()
        inp = U.to_nchw(U.preprocess(frame, img_size))
        outs = engine.infer(inp)
        boxes, scores, cls_ids = U.postprocess(outs, img_size, (w, h), conf_thr=args.conf,
                                               iou_thr=args.iou, reg_max=reg_max)
        dt = (time.perf_counter() - t0) * 1000
        vis = U.draw(frame, boxes, scores, cls_ids, class_names)
        save_path = os.path.join(args.save_dir, 'result.jpg')
        cv2.imwrite(save_path, vis)
        print(f'推論完成! 耗時: {dt:.2f} ms, 偵測到 {len(boxes)} 個物件, 存至 {save_path}')

if __name__ == '__main__':
    main()
"""
with open(f'{EXPORT_DIR}/jetson_infer.py', 'w', encoding='utf-8') as f:
    f.write(infer_py_content)

print('✅ 已生成 Jetson 專用部署套件至:', EXPORT_DIR)


### 6.1 Jetson Orin NX 最佳化技巧
- **供電與散熱模式**：
  Orin NX 具備動態電源管理，實機推論前建議啟用最高功率模式（MAXN / 25W）：
  ```bash
  sudo nvpmodel -m 0        # 啟用 25W / MAXN 效能模式
  sudo jetson_clocks        # 鎖定 GPU 與 CPU 核心最高頻率，消除延遲抖動
  ```
- **TensorRT 精度選擇**：
  - **FP16**（強烈建議）：Orin NX 配備強大的 Ampere Tensor Cores，FP16 模式可達到數百 FPS 推論速度，且精度幾乎零損耗。
  - **INT8**：若有極限高併發多路串流需求，可在板端搭配 `--int8` 與校正資料集量化。

In [ ]:
# 顯示即將打包至 Jetson 的部署檔案
print('即將部署至 Jetson 的檔案清單：')
for f in sorted(os.listdir(EXPORT_DIR)):
    p = f'{EXPORT_DIR}/{f}'
    if os.path.isfile(p):
        print(f'  - {f:35s} ({os.path.getsize(p) / 1024:.1f} KB)')

## 7. 打包輸出（Jetson 部署包）
`export/` 資料夾（在 Google Drive）會包含：

| 檔案 | 用途 |
|---|---|
| `*.onnx` | FP32 ONNX 模型，用於在 Jetson 上編譯 TensorRT Engine |
| `build_engine.sh` | 板端一鍵使用 `trtexec` 編譯 Engine 的腳本 |
| `jetson_infer.py` | Jetson 實機推論範例程式（支援圖片/相機/影片） |
| `damo_trt_utils.py` | 實機專用的前處理、後處理與 TensorRT 推論封裝 |
| `model_info.json` | 模型輸入尺寸、類別名稱、輸出特徵維度等配置 |
| `labels.txt` | 類別名稱清單（一行一個類別） |
| `*.pth` | PyTorch 權重檔（可用於後續微調） |

In [ ]:
# @title 打包與下載 { display-mode: "form" }
DOWNLOAD_ZIP = False  # @param {type:"boolean"}
import json, torch

ck = torch.load(FINAL_CKPT, map_location='cpu', weights_only=False)
torch.save({'model': ck['model']}, f'{EXPORT_DIR}/{EXP_NAME}.pth')

info = {
    'model_name': EXP_NAME,
    'model_size': MODEL_SIZE,
    'img_size': IMG_SIZE,
    'reg_max': REG_MAX,
    'strides': [8, 16, 32],
    'num_classes': len(CLASS_NAMES),
    'class_names': CLASS_NAMES,
    'onnx_file': f'{EXP_NAME}_{IMG_SIZE}.onnx',
    'precision_target': TRT_PRECISION,
    'input_format': '1x3xSxS RGB (float32, 0~255); plain resize, no letterbox, no mean/std',
    'outputs': ['cls_8', 'reg_8', 'cls_16', 'reg_16', 'cls_32', 'reg_32'],
    'output_note': 'cls: 1xKxHxW sigmoid scores; reg: 1x(4*(reg_max+1))xHxW DFL logits for (l,t,r,b)',
}
json.dump(info, open(f'{EXPORT_DIR}/model_info.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
open(f'{EXPORT_DIR}/labels.txt', 'w', encoding='utf-8').write('\n'.join(CLASS_NAMES) + '\n')

print('輸出資料夾：', EXPORT_DIR)
for f in sorted(os.listdir(EXPORT_DIR)):
    p = f'{EXPORT_DIR}/{f}'
    if os.path.isfile(p):
        print(f'  {f:40s} {os.path.getsize(p) / 1e6:8.2f} MB')

if DOWNLOAD_ZIP:
    from google.colab import files
    z = shutil.make_archive(f'/content/{EXP_NAME}_jetson_export', 'zip', EXPORT_DIR)
    files.download(z)

## 8. 板端部署與實機轉換（NVIDIA Jetson Orin NX）

完成訓練並下載部署包到 Jetson Orin NX 後，請依照以下步驟操作：

---

### 1. 將檔案複製至 Jetson Orin NX
將 Google Drive 的 `export/` 資料夾（或下載的 zip 解壓後）傳送至 Jetson 實機：
```bash
# 範例：使用 scp 從電腦傳送至 Jetson
scp -r ./export/* jetson@<JETSON_IP>:~/damoyolo_deploy/
```

---

### 2. 開啟 Orin NX 最大效能模式
```bash
# 設定為 MAXN / 25W 模式
sudo nvpmodel -m 0

# 鎖定 GPU / CPU 核心最高頻率
sudo jetson_clocks
```

---

### 3. 一鍵編譯 TensorRT Engine
進入部署資料夾，使用 `build_engine.sh` 或直接執行 `trtexec`：

```bash
cd ~/damoyolo_deploy

# 方法 A：使用內建腳本（預設 FP16 模式）
bash build_engine.sh
# 格式：bash build_engine.sh <ONNX檔名> <輸出Engine檔名> <fp16/int8>

# 方法 B：直接調用 trtexec
/usr/src/tensorrt/bin/trtexec \
    --onnx=my_damoyolo_S_640.onnx \
    --saveEngine=my_damoyolo_S_640.engine \
    --fp16 \
    --memPoolSize=workspace:2048
```
> **為什麼要在 Jetson 實機編譯？**
> TensorRT Engine (`.engine`) 與 GPU 硬體架構（Orin NX 為 Ampere SM 8.7）及 TensorRT 版本強烈綁定。在 Jetson 上透過 `trtexec` 編譯能確保針對 Orin NX 的 Tensor Core 進行最佳 kernel 融合與硬體排程。

---

### 4. 安裝 Python 執行套件
JetPack 系統已內建 CUDA、TensorRT 與 OpenCV。通常只需補充：
```bash
pip3 install pycuda numpy opencv-python
```

---

### 5. 執行實機推論測試

```bash
# 測試單張圖片
python3 jetson_infer.py --engine my_damoyolo_S_640.engine --source test.jpg --save-dir results

# 測試資料夾內所有圖片
python3 jetson_infer.py --engine my_damoyolo_S_640.engine --source ./test_images/

# USB 攝影機即時推論 (攝影機編號 0)
python3 jetson_infer.py --engine my_damoyolo_S_640.engine --source 0

# 測試影片檔
python3 jetson_infer.py --engine my_damoyolo_S_640.engine --source video.mp4
```

---

### 6. C++ 部署或 DeepStream 整合建議
- **C++ TensorRT Runtime**：可參考 TensorRT 官方 C++ 範例，將 `damo_trt_utils.py` 的解碼後處理（softmax + weighted sum + NMS）以 C++ / CUDA 實作，可進一步降低延遲至 3ms 內。
- **NVIDIA DeepStream SDK**：若有大規模多路攝影機串流需求，可編寫自訂後處理 parser 接入 DeepStream 的 `nvinferserver` 或 `nvinfer` 外掛。